# EVOLVEMEM: Self-Evolving Memory Architecture via AutoResearch for LLM Agents
**ArXivist-generated reproduction notebook**  
Paper: [arXiv:2605.13941](https://arxiv.org/abs/2605.13941)  
Authors: Jiaqi Liu, Xinyu Ye, Peng Xia, Zeyu Zheng, Cihang Xie, Mingyu Ding, Huaxiu Yao  
Generated: 2026-05-15

This notebook walks through the key components of the EVOLVEMEM implementation:
- Layer 1: Structured Memory Store (Section 3.1)
- Layer 2: Multi-View Retrieval + Fusion (Section 3.2)
- Layer 3: Self-Evolution Loop (Section 3.3, Algorithm 1)

All demos use **synthetic data** — no downloads required. End-to-end in ~5 minutes.

In [ ]:
# Cell 1: Environment check
import sys
print(f'Python: {sys.version}')
print(f'Platform: {sys.platform}')

# Check SQLite version (3.35+ required for FTS5)
import sqlite3
print(f'SQLite: {sqlite3.sqlite_version}')
assert tuple(int(x) for x in sqlite3.sqlite_version.split('.')) >= (3, 35, 0), \
    'SQLite 3.35+ required for FTS5 support'

# Check key packages
try:
    import numpy as np
    print(f'NumPy: {np.__version__}')
except ImportError:
    print('WARNING: numpy not found. Run: pip install numpy')

try:
    import yaml
    print('PyYAML: OK')
except ImportError:
    print('WARNING: pyyaml not found. Run: pip install pyyaml')

print('\nNote: sentence-transformers and openai/anthropic packages needed for full runs.')
print('This notebook uses HashingEmbedder + mock LLM to run without API keys.')

In [ ]:
# Cell 2: Install package in editable mode
import subprocess, sys, os

# Navigate to repo root (one level up from notebooks/)
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, repo_root)

try:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', repo_root, '-q'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('Package installed successfully.')
    else:
        print('Install stderr:', result.stderr[:300])
except Exception as e:
    print(f'Install failed: {e}. Add {repo_root} to sys.path manually.')

## Paper Overview

**Problem:** LLM agent memory systems grow their stored knowledge over time, but keep their *retrieval infrastructure* frozen. A retrieval policy calibrated for a small memory store becomes suboptimal as memories grow, and different question types (factual lookup vs. temporal vs. multi-hop adversarial) need fundamentally different retrieval strategies.

**Core innovation:** EVOLVEMEM makes the *retrieval configuration itself* a first-class optimization target. Instead of hand-tuning parameters, it runs an LLM-powered AutoResearch loop:

1. **EVALUATE**: Run all QA pairs with current config → per-question failure log
2. **DIAGNOSE**: LLM reads failure log, identifies root causes, proposes config delta
3. **PROPOSE**: Apply delta (clamped to safe ranges)
4. **GUARD**: Auto-revert if performance regresses; explore if stagnating

Starting from a minimal BM25-only baseline (R0: 30.5% F1), the loop discovers semantic retrieval, entity-swap, query decomposition, and answer verification — reaching 54.3% F1 at R7 on LoCoMo (+78% relative).

**Mapping to code:**
| Paper section | Code file |
|---|---|
| Section 3.1 Memory Store | `src/evolvemem/memory/store.py` |
| Section 3.1 Extractor | `src/evolvemem/memory/extractor.py` |
| Section 3.1 Consolidation | `src/evolvemem/memory/consolidator.py` |
| Section 3.2 Retriever + Eq. 1–5 | `src/evolvemem/retrieval/retriever.py` |
| Section 3.2 Action space Eq. 2 | `src/evolvemem/retrieval/config.py` |
| Section 3.3 + Algorithm 1 | `src/evolvemem/evolution/engine.py` |
| Appendix F Diagnosis prompt | `src/evolvemem/evolution/diagnosis.py` |

## Layer 1: Structured Memory Store (Section 3.1)

Each memory unit is a tuple $m = (c, \mu, \mathbf{e}, \eta)$ where:
- $c$: natural-language content
- $\mu \in \mathcal{T}$: type from six-category taxonomy (episodic, semantic, preference, ...)
- $\mathbf{e} \in \mathbb{R}^d$: dense embedding
- $\eta$: metadata (importance $\iota_i$, reinforcement score $\rho_i$, entities, timestamp)

The store uses SQLite/FTS5 with three retrieval indices: FTS5 for BM25, BLOB for semantic, and JSON fields for structured metadata.

In [ ]:
# Cell 5: Memory store demo
import sys, os
sys.path.insert(0, os.path.abspath('..'))

try:
    from src.evolvemem.memory.store import MemoryUnit, MemoryStore
    from src.evolvemem.embeddings.encoder import HashingEmbedder

    # Use in-memory SQLite for demo
    embedder = HashingEmbedder(dim=64)
    store = MemoryStore(db_path=':memory:', embedder=embedder)

    # Create sample memories (mimicking LoCoMo conversation content)
    memories = [
        MemoryUnit(
            content='Alice got promoted to senior software engineer at her company in March 2024.',
            memory_type='episodic',
            persons=['Alice'],
            topics=['career', 'promotion'],
            keywords=['promoted', 'senior', 'engineer'],
            importance=0.8,
            timestamp='2024-03-15',
        ),
        MemoryUnit(
            content='Alice prefers Python over Java for backend development.',
            memory_type='preference',
            persons=['Alice'],
            topics=['programming', 'preferences'],
            keywords=['Python', 'Java', 'backend'],
            importance=0.6,
        ),
        MemoryUnit(
            content='Bob and Alice went camping in Yosemite National Park in June 2024.',
            memory_type='episodic',
            persons=['Bob', 'Alice'],
            locations=['Yosemite National Park'],
            topics=['travel', 'camping'],
            keywords=['camping', 'Yosemite', 'hiking'],
            importance=0.7,
            timestamp='2024-06-10',
        ),
    ]

    store.add(memories)
    print(f'Store size: {store.size()} memory units')
    print(f'Memory types: {set(m.memory_type for m in store.get_all())}')
    print(f'\nSample unit: {store.get_all()[0]}')

except ImportError as e:
    print(f'Import error: {e}. Make sure you ran the install cell above.')

## Consolidation: Deduplication + Importance Decay (Section 3.1)

Three consolidation passes (Appendix A):
1. **Deduplication**: merge pairs with Jaccard similarity $\geq \tau_J = 0.80$, keep higher importance
2. **Importance decay**: $\iota_i \leftarrow \max(\iota_{\min}, \iota_i - \alpha_d \cdot t)$ where $\alpha_d = 0.05$/day, $\iota_{\min} = 0.15$
3. **Entity reinforcement**: $\rho_i \leftarrow \min(\rho_{\max}, \rho_i + \delta_\rho)$ when entities co-occur with query

In [ ]:
# Cell 7: Consolidation demo
try:
    from src.evolvemem.memory.consolidator import Consolidator

    consolidator = Consolidator(tau_j=0.80, alpha_d=0.05, iota_min=0.15)

    # Test deduplication: create a near-duplicate pair
    unit_a = MemoryUnit(
        content='Alice was promoted to senior engineer in March 2024.',
        memory_type='episodic', importance=0.9, persons=['Alice']
    )
    unit_b = MemoryUnit(
        content='Alice was promoted to senior engineer in March 2024 at her company.',
        memory_type='episodic', importance=0.6, persons=['Alice']
    )
    unit_c = MemoryUnit(
        content='Bob enjoys hiking and outdoor activities.',
        memory_type='preference', importance=0.7, persons=['Bob']
    )

    before = [unit_a, unit_b, unit_c]
    after = consolidator.deduplicate(before)

    print(f'Before dedup: {len(before)} units')
    print(f'After dedup:  {len(after)} units (merged near-duplicate a+b, kept higher importance)')
    for u in after:
        print(f'  - [{u.memory_type}] importance={u.importance:.2f}: {u.content[:60]}')

    # Test entity reinforcement
    query = 'What did Alice do for work?'
    reinforced = consolidator.reinforce_entities(after, query)
    print(f'\nAfter entity reinforcement with query "{query}":')
    for u in reinforced:
        print(f'  - rho={u.reinforcement_score:.3f}: {u.content[:60]}')

except Exception as e:
    print(f'Error: {e}')

## Layer 2: Multi-View Retrieval (Section 3.2)

Three retrieval views produce independent candidate sets:

**Lexical (BM25):** $s_{kw}(q, m_i) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,c_i)(k_1+1)}{f(t,c_i) + k_1(1-b+b|c_i|/|c|)}$

**Semantic (cosine):** $s_{sem}(q, m_i) = \cos(\mathbf{e}_q, \mathbf{e}_i) = \frac{\mathbf{e}_q^\top \mathbf{e}_i}{\|\mathbf{e}_q\| \|\mathbf{e}_i\|}$

**Final ranking (Equation 1):** $s(q, m_i; \theta) = s_{fuse} + \lambda_\iota \iota_i + \lambda_r \text{rec}(m_i) + \rho_i$

In [ ]:
# Cell 9: Multi-view retrieval demo
try:
    from src.evolvemem.retrieval.config import RetrievalConfig
    from src.evolvemem.retrieval.retriever import MultiViewRetriever

    # Use the store we created above
    config = RetrievalConfig(
        keyword_top_k=3,
        semantic_top_k=2,
        structured_top_k=2,
        max_context=3,
        fusion_mode='rrf',
    )

    retriever = MultiViewRetriever(store=store, config=config)

    # Test BM25 retrieval
    query = 'What is Alice\'s job?'
    bm25_results = store.search_bm25(query, top_k=3)
    print(f'BM25 results for "{query}":')
    for unit, score in bm25_results:
        print(f'  [{score:.3f}] {unit.content[:70]}')

    # Test semantic retrieval
    sem_results = store.search_semantic(query, top_k=3)
    print(f'\nSemantic results (HashingEmbedder — approximate):')
    for unit, score in sem_results:
        print(f'  [{score:.3f}] {unit.content[:70]}')

    # Full retrieval with RRF fusion
    retrieved = retriever.retrieve(query)
    print(f'\nFused retrieval (RRF, max_context={config.max_context}):')
    for i, unit in enumerate(retrieved, 1):
        print(f'  {i}. {unit.content[:70]}')

except Exception as e:
    print(f'Error: {e}')

## The Evolvable Action Space (Section 3.2 / Table 7)

The full configuration $\theta$ is defined in `RetrievalConfig`. The evolution engine can modify any dimension:

$$\theta = (k_{sem}, k_{kw}, k_{str}, B_{ctx}, \text{mode}, \{w_v\}, \alpha, \{\theta_c\}_{c \in C})$$

Starting from $\theta_0$ (BM25-only, minimal), the AutoResearch loop progressively activates dimensions.

In [ ]:
# Cell 11: Action space demo
try:
    from src.evolvemem.retrieval.config import RetrievalConfig

    # theta_0: minimal starting config (Section 4.1)
    theta_0 = RetrievalConfig.initial()
    print('theta_0 (initial, BM25-only):')
    print(f'  {theta_0}')

    # Simulate R1 delta: diagnosis proposes enabling semantic + RRF
    delta_r1 = {
        'semantic_top_k': 8,
        'keyword_top_k': 8,
        'structured_top_k': 3,
        'max_context': 16,
        'fusion_mode': 'rrf',
        'enable_entity_swap': False,
    }
    theta_1 = theta_0.apply_delta(delta_r1)
    print('\ntheta_1 (after R1 — semantic + RRF activated):')
    print(f'  {theta_1}')

    # Simulate R3 delta: entity-swap for adversarial category
    delta_r3 = {
        'per_category_overrides': {'5': {'enable_entity_swap': True}}
    }
    theta_3 = theta_1.apply_delta(delta_r3)
    print('\ntheta_3 (after R3 — entity-swap for Cat.5):')
    cat5_config = theta_3.for_category(5)
    print(f'  Global entity_swap: {theta_3.enable_entity_swap}')
    print(f'  Cat.5 entity_swap:  {cat5_config.enable_entity_swap}  (per-category override)')

    # Test clamping
    bad_config = RetrievalConfig(semantic_top_k=999, max_context=-5, fusion_mode='invalid')
    clamped = bad_config.clamp()
    print(f'\nClamped config (999→{clamped.semantic_top_k}, -5→{clamped.max_context}, invalid→{clamped.fusion_mode})')

except Exception as e:
    print(f'Error: {e}')

## Layer 3: Self-Evolution Loop (Section 3.3 / Algorithm 1)

The evolution objective (Equation 3):
$$\theta^* = \arg\max_{\theta \in \Theta} F(\theta; K, Q) = \frac{1}{|Q|} \sum_{(q,y^*) \in Q} \text{score}(\hat{y}(q; \theta, K), y^*)$$

The update rule (Equation 4):
$$\theta_{r+1} = \begin{cases} \theta^*_{r-1} & \text{if } f_{r-1} - f_r > \tau_{rev} \text{ (REVERT)} \\ \theta_r \oplus \eta_{exp} & \text{if stagnation × 2 rounds (EXPLORE)} \\ \text{clamp}_\Theta(\theta_r \oplus \Delta\theta_r) & \text{otherwise (APPLY)} \end{cases}$$

Below we simulate a 3-round evolution on a tiny synthetic QA set using a **mock diagnosis module** (no real LLM call needed).

In [ ]:
# Cell 13: Evolution loop demo with mock LLM
try:
    from src.evolvemem.retrieval.config import RetrievalConfig
    from src.evolvemem.evaluation.metrics import Evaluator

    # Mock diagnosis module that proposes a fixed delta each round
    class MockDiagnosis:
        def __init__(self):
            self.round = 0
            self._deltas = [
                {'semantic_top_k': 5, 'fusion_mode': 'rrf', 'max_context': 12},
                {'enable_entity_swap': True, 'structured_top_k': 3},
                {'enable_answer_verification': True},
            ]
        def diagnose(self, raw_log, current_config, memory_size):
            delta = self._deltas[min(self.round, len(self._deltas)-1)]
            self.round += 1
            return {'parameter_suggestions': delta, 'missing_topics': [], 'priority_actions': []}

    # Mock answer generator
    class MockAnswerGen:
        def __init__(self, retriever):
            self.retriever = retriever
        def generate(self, query, context, config, category=None):
            return context[0].content.split('.')[0] if context else 'unknown'

    # Synthetic QA pairs
    qa_pairs = [
        {'q': 'What is Alice\'s job?', 'ref': 'senior software engineer', 'category': 1},
        {'q': 'Where did Bob and Alice go camping?', 'ref': 'Yosemite National Park', 'category': 2},
        {'q': 'What programming language does Alice prefer?', 'ref': 'Python', 'category': 1},
    ]

    evaluator = Evaluator()
    mock_diagnosis = MockDiagnosis()
    mock_answer_gen = MockAnswerGen(retriever)

    theta = RetrievalConfig.initial()
    f_star = 0.0
    f_prev = 0.0
    epsilon = 0.005
    tau_rev = 0.01

    print('Simulating 3-round self-evolution loop...')
    print(f'{"Round":<8} {"F1":<10} {"Action"}')
    print('-' * 50)

    for r in range(3):
        # Evaluate
        scores = []
        for item in qa_pairs:
            retrieved = retriever.retrieve(item['q'], config=theta)
            pred = mock_answer_gen.generate(item['q'], retrieved, theta)
            scores.append(evaluator.token_f1(pred, item['ref']))
        f_r = sum(scores) / len(scores)

        # Diagnose
        proposal = mock_diagnosis.diagnose([], theta, store.size())
        delta = proposal.get('parameter_suggestions', {})

        # Guard logic
        regression = f_prev - f_r if r > 0 else 0.0
        if r > 0 and regression > tau_rev:
            action = 'REVERT'
        else:
            action = 'APPLY'
            theta = theta.apply_delta(delta)

        if f_r > f_star:
            f_star = f_r

        print(f'R{r:<7} {f_r:.4f}    {action}: {list(delta.keys())}')
        f_prev = f_r

    print(f'\nBest F1: {f_star:.4f}')
    print(f'Final config: {theta}')
    print('\n(Note: real runs use actual LLM calls and 1,986 QA pairs for 7 rounds)')

except Exception as e:
    import traceback
    print(f'Error: {e}')
    traceback.print_exc()

## Evaluation Metrics (Section 4.1)

LoCoMo uses **token-level F1**: $F1 = 2 \cdot \frac{|\hat{y} \cap y^*| / |\hat{y}| \cdot |\hat{y} \cap y^*| / |y^*|}{|\hat{y} \cap y^*| / |\hat{y}| + |\hat{y} \cap y^*| / |y^*|}$

MemBench uses exact-match accuracy.

In [ ]:
# Cell 15: Metrics demo
try:
    from src.evolvemem.evaluation.metrics import Evaluator

    ev = Evaluator()

    test_cases = [
        ('senior software engineer', 'senior software engineer', 'exact match'),
        ('Alice got promoted to senior engineer', 'Alice is a senior software engineer', 'partial match'),
        ('she works at a company', 'senior software engineer', 'wrong answer'),
        ('unknown', 'senior software engineer', 'abstention'),
    ]

    print(f'{"Prediction":<45} {"Reference":<35} {"F1":<8} {"Scenario"}')
    print('-' * 110)
    for pred, ref, scenario in test_cases:
        f1 = ev.token_f1(pred, ref)
        bleu = ev.bleu1(pred, ref)
        print(f'{pred[:44]:<45} {ref[:34]:<35} {f1:.3f}   {scenario}')

except Exception as e:
    print(f'Error: {e}')

## Paper Results

Results reported in the paper (Tables 2–3). To reproduce these, run the full evolution pipeline with GPT-4o/5.1 and the LoCoMo-10 / MemBench datasets.

In [ ]:
# Cell 17: Paper results summary
paper_results = {
    'LoCoMo (GPT-4o)': {
        'EVOLVEMEM overall F1': 0.543,
        'SimpleMem (best baseline) overall F1': 0.432,
        'relative gain over baseline': '25.7%',
        'baseline R0 F1': 0.305,
        'relative gain over R0': '78.0%',
    },
    'LoCoMo (GPT-5.1)': {
        'EVOLVEMEM overall F1': 0.572,
        'SimpleMem (best baseline) overall F1': 0.418,
        'relative gain over baseline': '36.8%',
    },
    'MemBench (GPT-4o)': {
        'EVOLVEMEM overall accuracy': 0.679,
        'best baseline (RecentMem/MemGPT)': 0.571,
        'relative gain': '18.9%',
    },
    'Cross-benchmark transfer': {
        'CL (evolved on LoCoMo, zero-shot to MemBench)': 0.543,
        'CLM (continued on MemBench)': 0.792,
        'CM (evolved on MemBench from scratch)': 0.679,
    },
}

for benchmark, results in paper_results.items():
    print(f'\n{benchmark}:')
    for k, v in results.items():
        print(f'  {k}: {v}')

print('\nTo reproduce: run evolve.py with full config, then evaluate.py')
print('Then use ArXivist Stage 6 Results Comparator to verify your numbers.')

## What To Do Next

1. **Set up your API key**: `export OPENAI_API_KEY=sk-...`
2. **Get the data**: See `data/README_data.md` for LoCoMo and MemBench download instructions
3. **Ingest conversations**: `python ingest.py --sessions-file data/locomo_sessions.jsonl --db-path evolvemem.db`
4. **Run evolution** (this is the main contribution): `python evolve.py --qa-file data/locomo_qa.jsonl --db-path evolvemem.db --output-config outputs/best_config.json`
5. **Evaluate**: `python evaluate.py --qa-file data/locomo_qa.jsonl --db-path evolvemem.db --evolved-config outputs/best_config.json`
6. **Compare results**: Feed `outputs/results.jsonl` to ArXivist Stage 6

**Key implementation assumptions** (see README for full list):
- `lambda_iota=1.0`, `lambda_r=1.0` (Eq.1 weights — not specified in paper) ⚠️ Medium confidence
- `tau_ver=0.5` (verification threshold — not specified in paper) ⚠️ Medium confidence  
- RRF k=60 (standard default — paper omits value) ✓ Low risk

**Diagnosis prompt quality is the most critical factor** for reproducing evolution gains.
The verbatim Appendix F prompts are implemented in `src/evolvemem/evolution/diagnosis.py`.